# 09. 리트리버 개선: 키워드 겹침 분석 + 쿼리 재작성 / HyDE

멘토링(11/01) 조언대로 **어디서 검색이 실패하는지 먼저 드러내고**, 그 구간을 올리는 기법을 **같은 조건에서** 비교합니다.
맨 위 `VERSION` 한 줄로 **v1.1 / v2.1**을 바꿔 돌릴 수 있습니다.

**1) 키워드 겹침 구간 분석**: 질문 단어 중 정답 문단에 나오는 비율로 시험 질문을 하·중·상 3구간으로 나누고, 구간별 성능을 봅니다.
그리고 실패를 "후보 100개에도 없음(검색 실패)"과 "후보에는 있는데 순위 밖(재정렬 실패)"으로 나눕니다.

**2) 비교하는 검색 질의** (모두 같은 bge 검색 100개 → 같은 MiniLM 재정렬. 재정렬은 항상 **원래 질문**으로 채점)

| 방법 | 검색에 쓰는 것 |
|---|---|
| Q (기준) | 원래 질문 |
| HyDE | Qwen이 쓴 "정답처럼 보이는 짧은 문단" |
| Q + HyDE | 원래 질문 + 가상 문단을 이어 붙인 글 (query2doc 방식) |
| 다중 쿼리 | 원래 질문 + 재작성 3개(의학 용어 / 쉬운 말 / 키워드)로 각각 검색 → RRF로 합침 |

- 가상 문단과 재작성은 **검색에만** 쓰고, 답변 근거로는 쓰지 않습니다.
- 최종 선택은 **검증 200**으로, 보고는 **시험 500**(답 있는 건강 질문)으로 합니다.
- 생성 결과는 드라이브에 저장돼서, 다시 돌리면 생성은 건너뜁니다.

**런타임: A100 (+ v2.1이면 High-RAM).** 위에서부터 Run all. 약 30~40분 (DB 불러오기 2~8분, Qwen 다운로드 약 5분, 생성 15~20분, 검색·재정렬 약 5분).
결과는 드라이브 `rag_msmarco/<버전 폴더>/query_rewrite/`와 `RAG/2_results/retrieval_rewrite_<버전>/`에 저장되고, 마지막 셀이 런타임을 끕니다.

## 1. 설치 + 버전 선택

In [ ]:
!pip -q install -U sentence-transformers faiss-cpu

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

VERSION = 'v2.1'          # 'v1.1' 또는 'v2.1'

SAVE_DIR   = {'v1.1': '/content/drive/MyDrive/rag_msmarco/v11_test',
              'v2.1': '/content/drive/MyDrive/rag_msmarco/health'}[VERSION]
OUT_DIR    = f'{SAVE_DIR}/query_rewrite'
REPORT_DIR = f'/content/drive/MyDrive/RAG/2_results/retrieval_rewrite_{VERSION}'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)
print(VERSION, '/ DB:', SAVE_DIR, '/ 결과:', OUT_DIR)

## 2. 설정

In [ ]:
import re, json, time, random, string
from collections import Counter
import numpy as np
import pandas as pd
import torch

SEED       = 42
K          = 100        # 1단계 후보 수 (= 재정렬할 후보 수)
RRF_K      = 60
GEN_BATCH  = 32
EMB_MODEL  = 'BAAI/bge-base-en-v1.5'
QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '
RERANKER   = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
LLM_NAME   = 'Qwen/Qwen2.5-7B-Instruct'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DEVICE == 'cuda', 'GPU 런타임이 아닙니다. 런타임 유형을 A100으로 바꾸세요.'
print('GPU:', torch.cuda.get_device_name(0))

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f]

def save_jsonl(rows, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

## 3. DB + 검색기 불러오기

시험 = 평가셋 중 **답 있는 건강 질문** (정답 문단이 있는 것). 검증 = 03에서 만든 `rerank/val.jsonl` 200개.

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

t = time.time()
corpus   = load_jsonl(f'{SAVE_DIR}/corpus.jsonl')
eval_set = load_jsonl(f'{SAVE_DIR}/eval.jsonl')
val_set  = load_jsonl(f'{SAVE_DIR}/rerank/val.jsonl')
index    = faiss.read_index(f'{SAVE_DIR}/faiss_bge_flatip.index')
texts    = [c['text'] for c in corpus]
assert index.ntotal == len(corpus), '인덱스와 문단 수가 다릅니다'

test_set = [q for q in eval_set if q['domain'] == 'health' and q.get('answerable', True) and q.get('gold_pids')]
val_set  = [q for q in val_set if q.get('gold_pids')]
print(f'문단 {len(corpus):,} / 시험 {len(test_set)} / 검증 {len(val_set)} ({time.time() - t:.0f}초)')

emb_model = SentenceTransformer(EMB_MODEL, device=DEVICE)
emb_model.half()
reranker = CrossEncoder(RERANKER, max_length=256, device=DEVICE)
reranker.model.half()

def encode(strings, prefix=True):
    pre = QUERY_PREFIX if prefix else ''
    return emb_model.encode([pre + s for s in strings], batch_size=128, normalize_embeddings=True,
                            convert_to_numpy=True).astype('float32')

def dense_search(strings, prefix=True):
    _, ids = index.search(encode(strings, prefix), K)
    return ids.tolist()

def rrf(lists, k=RRF_K, top=K):
    score = Counter()
    for lst in lists:
        for r, p in enumerate(lst):
            score[p] += 1 / (k + r + 1)
    return [p for p, _ in score.most_common(top)]

def rerank(queries, cand_lists):
    # 원래 질문으로 후보를 다시 채점해서 순서를 바꿈
    pairs = [(q, texts[p]) for q, c in zip(queries, cand_lists) for p in c]
    scores = reranker.predict(pairs, batch_size=512, show_progress_bar=len(pairs) > 50000)
    out, i = [], 0
    for c in cand_lists:
        s = scores[i:i + len(c)]; i += len(c)
        out.append([c[j] for j in np.argsort(-s)])
    return out

def first_rank(ranked, gold):
    gold = set(gold)
    for r, p in enumerate(ranked, 1):
        if p in gold:
            return r
    return None      # 후보 100개 안에 없음

def metrics(ranks):
    ranks = [r if r is not None else 10**9 for r in ranks]
    return {'R@1': np.mean([r <= 1 for r in ranks]), 'R@5': np.mean([r <= 5 for r in ranks]),
            'R@10': np.mean([r <= 10 for r in ranks]), 'R@100': np.mean([r <= 100 for r in ranks]),
            'MRR@10': np.mean([1 / r if r <= 10 else 0 for r in ranks])}

## 4. 키워드 겹침 구간 나누기

**겹침** = 질문의 내용어(불용어 제외) 중 정답 문단에 나오는 단어의 비율 (정답 문단이 여러 개면 가장 높은 것).
두 버전을 같은 기준으로 비교하려고 **고정 경계**를 씁니다: **하** = 절반 이하 겹침 (≤ 0.5) / **중** = 0.5 초과 ~ 1 미만 / **상** = 질문 단어가 정답 문단에 전부 나옴 (1.0).
(MS MARCO는 Bing 검색 결과로 만들어져서 '상'이 30~40%나 됩니다. 멘토가 말한 '평가가 쉬운 이유'가 이 비율입니다.)

In [ ]:
STOP = set('''a an the of to in on for and or is are was were be been being do does did what which who whom whose
when where why how can could should would will shall may might must i you he she it we they me my your his her its our
their this that these those there here with without from by at as about into than then so if not no yes any some
much many more most less also just only very too long get gets got have has had having'''.split())

def content_words(s):
    s = ''.join(ch if ch not in string.punctuation else ' ' for ch in s.lower())
    return [w for w in s.split() if w not in STOP]

def overlap(q):
    qw = set(content_words(q['query']))
    if not qw:
        return 0.0
    return max(len(qw & set(content_words(texts[p]))) / len(qw) for p in q['gold_pids'])

for q in test_set:
    q['overlap'] = overlap(q)
bucket_of = lambda o: '하' if o <= 0.5 else ('상' if o >= 0.999 else '중')
for q in test_set:
    q['bucket'] = bucket_of(q['overlap'])
n_b = Counter(q['bucket'] for q in test_set)
print('구간 (하 ≤ 0.5 < 중 < 1.0 = 상):', {b: f'{n_b[b]}개 ({n_b[b] / len(test_set):.0%})' for b in ['하', '중', '상']})
for b in ['하', '상']:
    print(f'\n[{b}] 예시')
    for q in [q for q in test_set if q['bucket'] == b][:3]:
        print(f"  ({q['overlap']:.2f}) {q['query']}  →  {texts[q['gold_pids'][0]][:110]}")

## 5. Qwen으로 가상 문단(HyDE) + 재작성 만들기

검증 + 시험 질문 전부에 대해 한 번만 생성하고 드라이브에 저장합니다 (`query_rewrite/generations.jsonl`). 이미 있으면 불러옵니다.

In [ ]:
GEN_PATH = f'{OUT_DIR}/generations.jsonl'
all_q = {q['query'] for q in test_set + val_set}
gens = {g['query']: g for g in load_jsonl(GEN_PATH)} if os.path.exists(GEN_PATH) else {}
todo = sorted(all_q - set(gens))
print(f'생성할 질문 {len(todo)}개 (저장된 것 {len(gens)}개)')

if todo:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(LLM_NAME, padding_side='left')
    llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, dtype=torch.bfloat16, device_map='cuda')
    llm.eval()

    SYS_HYDE = ('You write short passages for a health information website. Given a question, write one passage of '
                '3-4 sentences (about 60 words) that directly answers it, in the style of a web page. '
                'Output only the passage.')
    SYS_RW = ('Rewrite the health question into 3 different web search queries:\n'
              '1. using precise medical terminology\n2. in plain everyday words\n3. a short keyword query\n'
              'Output exactly 3 lines, one query per line, with no numbering or extra text.')

    @torch.inference_mode()
    def generate(system, user_list, max_new):
        outs = []
        for i in range(0, len(user_list), GEN_BATCH):
            prompts = [tok.apply_chat_template([{'role': 'system', 'content': system}, {'role': 'user', 'content': u}],
                                               tokenize=False, add_generation_prompt=True) for u in user_list[i:i + GEN_BATCH]]
            enc = tok(prompts, return_tensors='pt', padding=True).to(llm.device)
            gen = llm.generate(**enc, max_new_tokens=max_new, do_sample=False, temperature=None, top_p=None, top_k=None,
                               pad_token_id=tok.pad_token_id)
            outs += [o.strip() for o in tok.batch_decode(gen[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)]
            print(f'  {min(i + GEN_BATCH, len(user_list))}/{len(user_list)}', end='\r')
        return outs

    t = time.time()
    hyde = generate(SYS_HYDE, todo, 120)
    print(f'\nHyDE 끝 ({time.time() - t:.0f}초)')
    t = time.time()
    rw_raw = generate(SYS_RW, todo, 80)
    print(f'\n재작성 끝 ({time.time() - t:.0f}초)')

    for q, h, r in zip(todo, hyde, rw_raw):
        lines = [re.sub(r'^\s*(\d+[.)]|[-*])\s*', '', l).strip() for l in r.split('\n')]
        gens[q] = {'query': q, 'hyde': h, 'rewrites': [l for l in lines if l][:3]}
    save_jsonl(list(gens.values()), GEN_PATH)
    del llm; torch.cuda.empty_cache()
    print('저장:', GEN_PATH)

ex = gens[test_set[0]['query']]
print('\n예시 질문:', ex['query'])
print('HyDE   :', ex['hyde'])
print('재작성 :', ex['rewrites'])

## 6. 방법별 검색 + 재정렬

각 방법으로 후보 100개를 뽑고(재정렬 전), 원래 질문으로 MiniLM 재정렬(재정렬 후)을 합니다.

In [ ]:
def run_methods(qset):
    queries = [q['query'] for q in qset]
    hy = [gens[s]['hyde'] for s in queries]
    cands = {
        'Q (기준)':  dense_search(queries),
        'HyDE':      dense_search(hy, prefix=False),
        'Q + HyDE':  dense_search([f'{s} {h}' for s, h in zip(queries, hy)]),
    }
    variants = [[s] + gens[s]['rewrites'] for s in queries]
    flat = [v for vs in variants for v in vs]
    flat_ids = dense_search(flat)
    i, mq = 0, []
    for vs in variants:
        mq.append(rrf(flat_ids[i:i + len(vs)])); i += len(vs)
    cands['다중 쿼리'] = mq

    out = {}
    for name, c in cands.items():
        rr = rerank(queries, c)
        out[name] = {'before': [first_rank(x, q['gold_pids']) for x, q in zip(c, qset)],
                     'after':  [first_rank(x, q['gold_pids']) for x, q in zip(rr, qset)]}
        print(f'  {name} 끝')
    return out

t = time.time()
print('검증'); val_res = run_methods(val_set)
print('시험'); test_res = run_methods(test_set)
print(f'{time.time() - t:.0f}초')

def table_of(res, stage):
    return pd.DataFrame({name: metrics(r[stage]) for name, r in res.items()}).T.round(3)

val_after = table_of(val_res, 'after')
BEST = val_after['MRR@10'].idxmax()
print('\n[검증] 재정렬 후'); display(val_after)
print('검증 MRR@10 기준 최종 선택:', BEST)

test_before, test_after = table_of(test_res, 'before'), table_of(test_res, 'after')
print('\n[시험] 재정렬 전 (dense 후보만)'); display(test_before)
print('[시험] 재정렬 후'); display(test_after)

## 7. 겹침 구간별 결과 + 실패 유형

- **구간별 R@10 / MRR@10** (재정렬 후): 겹침이 낮은 구간을 어떤 방법이 올리는지
- **실패 유형** (Q 기준, 재정렬 후 상위 10개에 정답이 없는 질문): 후보 100개에도 없음(검색 실패) vs 후보에는 있음(재정렬 실패)

In [ ]:
buckets = ['하', '중', '상']
b_idx = {b: [i for i, q in enumerate(test_set) if q['bucket'] == b] for b in buckets}

rows = []
for name, r in test_res.items():
    for b in buckets:
        m = metrics([r['after'][i] for i in b_idx[b]])
        rows.append({'방법': name, '구간': b, 'R@10': m['R@10'], 'MRR@10': m['MRR@10'], 'R@100': m['R@100']})
by_bucket = pd.DataFrame(rows)
r10 = by_bucket.pivot(index='방법', columns='구간', values='R@10')[buckets].round(3)
mrr = by_bucket.pivot(index='방법', columns='구간', values='MRR@10')[buckets].round(3)
print('구간별 R@10 (재정렬 후)'); display(r10)
print('구간별 MRR@10 (재정렬 후)'); display(mrr)

fail = []
for b in buckets:
    ranks = [test_res['Q (기준)']['after'][i] for i in b_idx[b]]
    n = len(ranks)
    fail.append({'구간': b, '질문 수': n,
                 '상위 10 안 (성공)': sum(r is not None and r <= 10 for r in ranks) / n,
                 '후보에는 있음 (순위 밖)': sum(r is not None and r > 10 for r in ranks) / n,
                 '후보에도 없음 (검색 실패)': sum(r is None for r in ranks) / n})
fail = pd.DataFrame(fail).set_index('구간').round(3)
print('실패 유형 (Q 기준, 재정렬 후)'); display(fail)

In [ ]:
# 겹침 '하' 구간에서 Q는 실패했지만 다른 방법이 성공한 예 (발표·사례 분석용)
def show(i):
    q = test_set[i]; g = gens[q['query']]
    print(f"\n({q['overlap']:.2f}) {q['query']}")
    print('  정답 문단 :', texts[q['gold_pids'][0]][:160])
    print('  HyDE      :', g['hyde'][:160])
    print('  재작성    :', g['rewrites'])
    print('  순위      :', {name: r['after'][i] for name, r in test_res.items()})

wins = [i for i in b_idx['하'] if (test_res['Q (기준)']['after'][i] or 999) > 10
        and any((r['after'][i] or 999) <= 10 for n, r in test_res.items() if n != 'Q (기준)')]
print(f"겹침 '하' 구간에서 Q 실패 → 다른 방법 성공: {len(wins)}개")
for i in wins[:5]:
    show(i)

## 8. 저장

In [ ]:
stamp = time.strftime('%Y-%m-%d %H:%M', time.localtime(time.time() + 9 * 3600))
summary = '\n'.join([
    f'# 09 리트리버 개선: 겹침 분석 + 쿼리 재작성 / HyDE ({VERSION}) ({stamp} KST)', '',
    f'- DB: 문단 {len(corpus):,}개 / 시험 {len(test_set)} (답 있는 건강) / 검증 {len(val_set)}',
    f'- 검색: bge 후보 {K}개 → MiniLM-L-6 재정렬 (원래 질문으로 채점) / 생성: {LLM_NAME}',
    f'- 겹침 구간: 하 ≤ 0.5 < 중 < 1.0 = 상 (질문 내용어 중 정답 문단에 나오는 비율) / '
    f'하 {n_b["하"]} · 중 {n_b["중"]} · 상 {n_b["상"]}개',
    f'- 검증 MRR@10 기준 최종 선택: **{BEST}**', '',
    '## 시험: 재정렬 전 (dense 후보만)', '', test_before.to_markdown(), '',
    '## 시험: 재정렬 후', '', test_after.to_markdown(), '',
    '## 검증: 재정렬 후', '', val_after.to_markdown(), '',
    '## 겹침 구간별 R@10 (재정렬 후)', '', r10.to_markdown(), '',
    '## 겹침 구간별 MRR@10 (재정렬 후)', '', mrr.to_markdown(), '',
    '## 실패 유형 (Q 기준, 재정렬 후)', '', fail.to_markdown(),
])
per_q = [{'qid': q['qid'], 'query': q['query'], 'overlap': round(q['overlap'], 3), 'bucket': q['bucket'],
          'gold_pids': q['gold_pids'],
          **{f'rank_{name}': r['after'][i] for name, r in test_res.items()},
          **{f'rank_before_{name}': r['before'][i] for name, r in test_res.items()}}
         for i, q in enumerate(test_set)]
for d in (OUT_DIR, REPORT_DIR):
    with open(f'{d}/results_summary.md', 'w', encoding='utf-8') as f:
        f.write(summary)
    save_jsonl(per_q, f'{d}/per_question.jsonl')
print(summary)
print('\n저장:', OUT_DIR, '/', REPORT_DIR)

## 9. 드라이브 저장 마무리 + 런타임 종료

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
runtime.unassign()